## How Do Computers Solve Math?

When you type `(100 + 200) * 3` into a calculator, you instinctively know to add first, then multiply. But **computers don't think like humans** — they need a systematic algorithm to handle:

- **Operator precedence** (`*` before `+`)
- **Parentheses** (grouping overrides precedence)
- **Left-to-right evaluation**

The solution? **Reverse Polish Notation (RPN)** — a technique invented by Polish mathematician Jan Łukasiewicz and popularized by Hewlett-Packard calculators in the 1970s.

### The Pipeline

```
Expression String → Tokenize → Convert to RPN → Evaluate → Result
"(100 + 200) * 3" → [100, +, 200, *, 3] → [100, 200, +, 3, *] → 900
```

### Key Data Structures

| Structure | Used For | Why |
|-----------|----------|-----|
| **ArrayList** | Storing tokens & RPN output | Ordered sequence, index access |
| **Stack** | Operator reordering & calculation | LIFO — last operator in is first out |
| **HashMap** | Looking up operators by character | O(1) lookup for precedence & calculation |

Let's build each piece step by step.

---

## Step 1: The Token Class

Every part of a math expression is a **Token** — a number, an operator (`+`, `-`, `*`, `/`), or a parenthesis.

The `Token` class stores:
- The **character** (e.g. `+`, `*`)
- The **precedence** (multiplication before addition)
- The **calculation** function (what the operator actually does)

**Key Concept:** We use Java's `BiFunction<Double, Double, Double>` — a functional interface that takes two doubles and returns a double. This lets us store math operations as data.

### Challenge: Read the Token class below, then modify it to add a `POWER` operator check.

Run the code to see the test output.

In [ ]:
import java.util.function.BiFunction;

public class Token {
    private final Character token;
    private final int precedence;
    private final BiFunction<Double, Double, Double> calculation;
    private final int numArgs;

    // Constructor for passive/non-operator tokens
    public Token() { this('0'); }

    // Constructor for simple token (parenthesis)
    public Token(Character token) { this(token, 0, null, 0); }

    // Full constructor for operators
    public Token(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        this.token = token;
        this.precedence = precedence;
        this.calculation = calculation;
        this.numArgs = numArgs;
    }

    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }

    // Returns true if THIS token has higher precedence (lower number = higher priority)
    public boolean isPrecedent(Token other) {
        return this.precedence > other.getPrecedence();
    }

    // Execute the math operation
    public Double calculate(Double a, Double b) {
        return this.calculation.apply(a, b);
    }

    public String toString() { return this.token.toString(); }
}

// --- TEST ---
Token add = new Token('+', 4, (a, b) -> a + b, 2);
Token mul = new Token('*', 3, (a, b) -> a * b, 2);
Token paren = new Token('(');

System.out.println("Token '+' precedence: " + add.getPrecedence());
System.out.println("Token '*' precedence: " + mul.getPrecedence());
System.out.println("Does '+' yield to '*'? " + add.isPrecedent(mul));
System.out.println("3 + 5 = " + add.calculate(3.0, 5.0));
System.out.println("3 * 5 = " + mul.calculate(3.0, 5.0));

// TODO: Create a POWER token ('^', precedence 2) and test 2^10


{% capture challenge_token %}
Read the Token class. It represents a single element in a math expression — an operator with its precedence and calculation function. The test creates a few tokens and checks precedence. Try modifying the test to create a POWER token with precedence 2.
{% endcapture %}

{% capture code_token %}
import java.util.function.BiFunction;

public class Token {
    private final Character token;
    private final int precedence;
    private final BiFunction<Double, Double, Double> calculation;
    private final int numArgs;

    // Constructor for passive/non-operator tokens
    public Token() { this('0'); }

    // Constructor for simple token (parenthesis)
    public Token(Character token) { this(token, 0, null, 0); }

    // Full constructor for operators
    public Token(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        this.token = token;
        this.precedence = precedence;
        this.calculation = calculation;
        this.numArgs = numArgs;
    }

    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }

    // Returns true if THIS token has higher precedence (lower number = higher priority)
    public boolean isPrecedent(Token other) {
        return this.precedence > other.getPrecedence();
    }

    // Execute the math operation
    public Double calculate(Double a, Double b) {
        return this.calculation.apply(a, b);
    }

    public String toString() { return this.token.toString(); }
}

// --- TEST ---
Token add = new Token('+', 4, (a, b) -> a + b, 2);
Token mul = new Token('*', 3, (a, b) -> a * b, 2);
Token paren = new Token('(');

System.out.println("Token '+' precedence: " + add.getPrecedence());
System.out.println("Token '*' precedence: " + mul.getPrecedence());
System.out.println("Does '+' yield to '*'? " + add.isPrecedent(mul));
System.out.println("3 + 5 = " + add.calculate(3.0, 5.0));
System.out.println("3 * 5 = " + mul.calculate(3.0, 5.0));

// TODO: Create a POWER token ('^', precedence 2) and test 2^10
{% endcapture %}

{% include code-runner.html
   runner_id="rpn_token"
   language="java"
   challenge=challenge_token
   code=code_token
   height="500px"
%}


<div class="rpn-popcorn">
    <div class="rpn-popcorn__header">
        <h3>Popcorn Hack 1: Precedence Check</h3>
        <p>In the Token class, lower precedence numbers mean higher priority. Given <code>Token add = new Token('+', 4, ...)</code> and <code>Token mul = new Token('*', 3, ...)</code>, what does <code>add.isPrecedent(mul)</code> return?</p>
    </div>
    <div class="rpn-popcorn__body">
        <div class="rpn-popcorn__options" id="ph1-options">
            <button onclick="checkPH1(0)" class="rpn-popcorn__option">A) true — because '+' comes first alphabetically</button>
            <button onclick="checkPH1(1)" class="rpn-popcorn__option">B) true — because '+' has a higher number (4 > 3), meaning it yields to '*'</button>
            <button onclick="checkPH1(2)" class="rpn-popcorn__option">C) false — because '+' has higher precedence than '*'</button>
            <button onclick="checkPH1(3)" class="rpn-popcorn__option">D) Compilation error — you can't compare Token objects</button>
        </div>
        <div id="ph1-feedback" class="rpn-popcorn__feedback"></div>
    </div>
</div>

<script>
function checkPH1(sel) {
    const btns = document.querySelectorAll('#ph1-options .rpn-popcorn__option');
    const fb = document.getElementById('ph1-feedback');
    btns.forEach(b => b.className = 'rpn-popcorn__option');
    if (sel === 1) {
        btns[1].classList.add('rpn-popcorn__option--correct');
        fb.className = 'rpn-popcorn__feedback rpn-popcorn__feedback--correct';
        fb.innerHTML = '<strong style="color:#2ecc71;">Correct!</strong> <code>isPrecedent</code> checks <code>this.precedence > other.getPrecedence()</code>. Since 4 > 3, it returns <strong>true</strong> — meaning <code>+</code> yields to <code>*</code> during the Shunting-Yard algorithm. Higher number = lower priority = yields.';
    } else {
        btns[sel].classList.add('rpn-popcorn__option--incorrect');
        fb.className = 'rpn-popcorn__feedback rpn-popcorn__feedback--incorrect';
        if (sel === 0) fb.innerHTML = '<strong style="color:#e74c3c;">Incorrect.</strong> Precedence has nothing to do with alphabetical order — it\'s determined by the integer value passed to the constructor.';
        if (sel === 2) fb.innerHTML = '<strong style="color:#e74c3c;">Incorrect.</strong> Actually, <code>+</code> has a <em>higher number</em> (4), which means <em>lower</em> priority. In math, <code>*</code> is evaluated before <code>+</code>.';
        if (sel === 3) fb.innerHTML = '<strong style="color:#e74c3c;">Incorrect.</strong> <code>isPrecedent</code> is a regular method that compares integer values — no compilation error.';
    }
}
</script>


---

## Step 2: TermOrOperator — Subclass of Token

A `TermOrOperator` extends `Token` to also represent **numeric values**. This is an example of **inheritance** — the key AP CSA concept.

- If it's a **number**: stores the value as a String (e.g. `"100"`, `"3.14"`)
- If it's an **operator**: inherits the token, precedence, and calculation from Token
- If it's a **parenthesis**: just stores the character

**Why inheritance?** Both numbers and operators need to live in the same ArrayList. By making them share a parent class, we can store `ArrayList<TermOrOperator>` and treat them polymorphically.

### Challenge: Run the code to see how different TermOrOperator types behave.

In [ ]:
import java.util.function.BiFunction;

// --- Token (parent class) ---
class Token {
    private final Character token;
    private final int precedence;
    private final BiFunction<Double, Double, Double> calculation;
    private final int numArgs;

    public Token() { this('0'); }
    public Token(Character token) { this(token, 0, null, 0); }
    public Token(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        this.token = token;
        this.precedence = precedence;
        this.calculation = calculation;
        this.numArgs = numArgs;
    }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return this.precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return this.calculation.apply(a, b); }
    public String toString() { return this.token.toString(); }
}

// --- TermOrOperator (child class) ---
class TermOrOperator extends Token {
    private final String value;

    // Constructor for numeric values
    public TermOrOperator(String value) {
        this.value = value;
    }

    // Constructor for parenthesis
    public TermOrOperator(Character token) {
        super(token);
        this.value = null;
    }

    // Constructor for binary operators
    public TermOrOperator(Character token, int precedence, BiFunction<Double, Double, Double> calculation) {
        super(token, precedence, calculation, 2);
        this.value = null;
    }

    // Constructor for unary operators (like sqrt)
    public TermOrOperator(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        super(token, precedence, calculation, numArgs);
        this.value = null;
    }

    public String getValue() { return value; }

    public String toString() {
        return (this.value != null) ? this.value : super.toString();
    }
}

// --- TEST ---
TermOrOperator num = new TermOrOperator("42");
TermOrOperator plus = new TermOrOperator('+', 4, (a, b) -> a + b);
TermOrOperator openParen = new TermOrOperator('(');

System.out.println("Number token: " + num + " (value=" + num.getValue() + ")");
System.out.println("Operator token: " + plus + " (precedence=" + plus.getPrecedence() + ")");
System.out.println("Parenthesis token: " + openParen);
System.out.println();
System.out.println("Polymorphism demo — all in one ArrayList:");
java.util.ArrayList<TermOrOperator> list = new java.util.ArrayList<>();
list.add(openParen);
list.add(num);
list.add(plus);
list.add(new TermOrOperator("8"));
list.add(new TermOrOperator(')'));
System.out.println(list);

// TODO: Create a modulo operator (%, precedence 3) and test 10 % 3


{% capture challenge_termop %}
TermOrOperator extends Token to also hold numeric values. Run the code to see the three types: value, operator, and parenthesis. Then try adding a new operator for modulo (%).
{% endcapture %}

{% capture code_termop %}
import java.util.function.BiFunction;

// --- Token (parent class) ---
class Token {
    private final Character token;
    private final int precedence;
    private final BiFunction<Double, Double, Double> calculation;
    private final int numArgs;

    public Token() { this('0'); }
    public Token(Character token) { this(token, 0, null, 0); }
    public Token(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        this.token = token;
        this.precedence = precedence;
        this.calculation = calculation;
        this.numArgs = numArgs;
    }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return this.precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return this.calculation.apply(a, b); }
    public String toString() { return this.token.toString(); }
}

// --- TermOrOperator (child class) ---
class TermOrOperator extends Token {
    private final String value;

    // Constructor for numeric values
    public TermOrOperator(String value) {
        this.value = value;
    }

    // Constructor for parenthesis
    public TermOrOperator(Character token) {
        super(token);
        this.value = null;
    }

    // Constructor for binary operators
    public TermOrOperator(Character token, int precedence, BiFunction<Double, Double, Double> calculation) {
        super(token, precedence, calculation, 2);
        this.value = null;
    }

    // Constructor for unary operators (like sqrt)
    public TermOrOperator(Character token, int precedence, BiFunction<Double, Double, Double> calculation, int numArgs) {
        super(token, precedence, calculation, numArgs);
        this.value = null;
    }

    public String getValue() { return value; }

    public String toString() {
        return (this.value != null) ? this.value : super.toString();
    }
}

// --- TEST ---
TermOrOperator num = new TermOrOperator("42");
TermOrOperator plus = new TermOrOperator('+', 4, (a, b) -> a + b);
TermOrOperator openParen = new TermOrOperator('(');

System.out.println("Number token: " + num + " (value=" + num.getValue() + ")");
System.out.println("Operator token: " + plus + " (precedence=" + plus.getPrecedence() + ")");
System.out.println("Parenthesis token: " + openParen);
System.out.println();
System.out.println("Polymorphism demo — all in one ArrayList:");
java.util.ArrayList<TermOrOperator> list = new java.util.ArrayList<>();
list.add(openParen);
list.add(num);
list.add(plus);
list.add(new TermOrOperator("8"));
list.add(new TermOrOperator(')'));
System.out.println(list);

// TODO: Create a modulo operator (%, precedence 3) and test 10 % 3
{% endcapture %}

{% include code-runner.html
   runner_id="rpn_termop"
   language="java"
   challenge=challenge_termop
   code=code_termop
   height="550px"
%}


<div class="rpn-popcorn">
    <div class="rpn-popcorn__header" style="background: linear-gradient(135deg, #8e44ad 0%, #9b59b6 100%); border-bottom-color: #2ecc71;">
        <h3 style="color: #2ecc71;">Popcorn Hack 2: Inheritance & Polymorphism</h3>
        <p>We store both numbers and operators in <code>ArrayList&lt;TermOrOperator&gt;</code>. Which AP CSA concept makes this possible?</p>
    </div>
    <div class="rpn-popcorn__body" style="background: #1a1a1a;">
        <div class="rpn-popcorn__grid" id="ph2-options">
            <div onclick="checkPH2(0)" class="rpn-popcorn__grid-card">
                <code>Encapsulation — private fields hide implementation</code>
            </div>
            <div onclick="checkPH2(1)" class="rpn-popcorn__grid-card">
                <code>Polymorphism — subclass objects stored as parent type</code>
            </div>
            <div onclick="checkPH2(2)" class="rpn-popcorn__grid-card">
                <code>Abstraction — Token is an abstract class</code>
            </div>
            <div onclick="checkPH2(3)" class="rpn-popcorn__grid-card">
                <code>Overloading — multiple constructors in Token</code>
            </div>
        </div>
        <div id="ph2-feedback" class="rpn-popcorn__feedback"></div>
    </div>
</div>

<script>
function checkPH2(sel) {
    const cards = document.querySelectorAll('#ph2-options .rpn-popcorn__grid-card');
    const fb = document.getElementById('ph2-feedback');
    cards.forEach(c => c.className = 'rpn-popcorn__grid-card');
    if (sel === 1) {
        cards[1].classList.add('rpn-popcorn__grid-card--correct');
        fb.className = 'rpn-popcorn__feedback rpn-popcorn__feedback--correct';
        fb.innerHTML = '<strong style="color:#2ecc71;">Correct!</strong> <strong>Polymorphism</strong> lets us declare <code>ArrayList&lt;TermOrOperator&gt;</code> and store different subtypes. When we call <code>toString()</code>, Java dynamically dispatches to the correct implementation — numbers print their value, operators print their character.';
    } else {
        cards[sel].classList.add('rpn-popcorn__grid-card--incorrect');
        fb.className = 'rpn-popcorn__feedback rpn-popcorn__feedback--incorrect';
        if (sel === 0) fb.innerHTML = '<strong style="color:#e74c3c;">Not quite.</strong> Encapsulation is present (private fields + getters), but it\'s not what allows storing different types in the same ArrayList.';
        if (sel === 2) fb.innerHTML = '<strong style="color:#e74c3c;">Not quite.</strong> Token is a concrete class, not abstract. And even if it were, the concept of storing subclass instances as parent type is still polymorphism.';
        if (sel === 3) fb.innerHTML = '<strong style="color:#e74c3c;">Not quite.</strong> Overloading means multiple constructors/methods with different signatures. The key concept for storing mixed types in one list is polymorphism.';
    }
}
</script>


---

## Step 3: Tokens Map — Fast Operator Lookup

The `Tokens` class uses a **HashMap** to store all operators and separators. This gives us **O(1) lookup** — when we encounter a character in the expression, we can instantly check:

1. Is it an operator? → `operators.contains('+')`
2. What's its precedence? → `operators.getPrecedence('+')`
3. How does it calculate? → `operators.get('+').calculate(3, 5)`

**AP CSA Connection:** This is a real-world use of `HashMap<K, V>` — one of the most important data structures.

### Challenge: Run the code and then add the `^` (power) operator to the map.

In [ ]:
import java.util.Map;
import java.util.HashMap;
import java.util.function.BiFunction;

// Compact Token + TermOrOperator for this runner
class Token {
    private final Character token;
    private final int precedence;
    private final BiFunction<Double, Double, Double> calculation;
    private final int numArgs;
    public Token() { this('0'); }
    public Token(Character token) { this(token, 0, null, 0); }
    public Token(Character token, int precedence, BiFunction<Double, Double, Double> calc, int numArgs) {
        this.token = token; this.precedence = precedence; this.calculation = calc; this.numArgs = numArgs;
    }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return this.precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return calculation.apply(a, b); }
    public String toString() { return token.toString(); }
}

class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String value) { this.value = value; }
    public TermOrOperator(Character token) { super(token); this.value = null; }
    public TermOrOperator(Character token, int prec, BiFunction<Double, Double, Double> calc) {
        super(token, prec, calc, 2); this.value = null;
    }
    public TermOrOperator(Character token, int prec, BiFunction<Double, Double, Double> calc, int n) {
        super(token, prec, calc, n); this.value = null;
    }
    public String getValue() { return value; }
    public String toString() { return value != null ? value : super.toString(); }
}

// --- Tokens Map (HashMap wrapper) ---
class Tokens {
    private Map<Character, TermOrOperator> map = new HashMap<>();

    public void put(Character token) {
        map.put(token, new TermOrOperator(token));
    }
    public void put(Character token, int prec, BiFunction<Double, Double, Double> calc) {
        map.put(token, new TermOrOperator(token, prec, calc));
    }
    public void put(Character token, int prec, BiFunction<Double, Double, Double> calc, int n) {
        map.put(token, new TermOrOperator(token, prec, calc, n));
    }
    public TermOrOperator get(Character token) { return map.get(token); }
    public boolean contains(Character token) { return map.containsKey(token); }
    public String toString() { return map.toString(); }
}

// --- TEST ---
Tokens ops = new Tokens();
ops.put('+', 4, (a, b) -> a + b);
ops.put('-', 4, (a, b) -> a - b);
ops.put('*', 3, (a, b) -> a * b);
ops.put('/', 3, (a, b) -> a / b);
ops.put('%', 3, (a, b) -> a % b);

System.out.println("Operator map: " + ops);
System.out.println("Contains '+'? " + ops.contains('+'));
System.out.println("Contains 'x'? " + ops.contains('x'));
System.out.println("Precedence of '+': " + ops.get('+').getPrecedence());
System.out.println("Precedence of '*': " + ops.get('*').getPrecedence());
System.out.println("10 / 3 = " + ops.get('/').calculate(10.0, 3.0));
System.out.println("10 % 3 = " + ops.get('%').calculate(10.0, 3.0));

// TODO: Add power operator (^, precedence 2, Math.pow) and test 2^8


{% capture challenge_tokens %}
The Tokens class uses a HashMap for O(1) operator lookup. Run to see all registered operators. Then add the power operator (^, precedence 2, Math.pow).
{% endcapture %}

{% capture code_tokens %}
import java.util.Map;
import java.util.HashMap;
import java.util.function.BiFunction;

// Compact Token + TermOrOperator for this runner
class Token {
    private final Character token;
    private final int precedence;
    private final BiFunction<Double, Double, Double> calculation;
    private final int numArgs;
    public Token() { this('0'); }
    public Token(Character token) { this(token, 0, null, 0); }
    public Token(Character token, int precedence, BiFunction<Double, Double, Double> calc, int numArgs) {
        this.token = token; this.precedence = precedence; this.calculation = calc; this.numArgs = numArgs;
    }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return this.precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return calculation.apply(a, b); }
    public String toString() { return token.toString(); }
}

class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String value) { this.value = value; }
    public TermOrOperator(Character token) { super(token); this.value = null; }
    public TermOrOperator(Character token, int prec, BiFunction<Double, Double, Double> calc) {
        super(token, prec, calc, 2); this.value = null;
    }
    public TermOrOperator(Character token, int prec, BiFunction<Double, Double, Double> calc, int n) {
        super(token, prec, calc, n); this.value = null;
    }
    public String getValue() { return value; }
    public String toString() { return value != null ? value : super.toString(); }
}

// --- Tokens Map (HashMap wrapper) ---
class Tokens {
    private Map<Character, TermOrOperator> map = new HashMap<>();

    public void put(Character token) {
        map.put(token, new TermOrOperator(token));
    }
    public void put(Character token, int prec, BiFunction<Double, Double, Double> calc) {
        map.put(token, new TermOrOperator(token, prec, calc));
    }
    public void put(Character token, int prec, BiFunction<Double, Double, Double> calc, int n) {
        map.put(token, new TermOrOperator(token, prec, calc, n));
    }
    public TermOrOperator get(Character token) { return map.get(token); }
    public boolean contains(Character token) { return map.containsKey(token); }
    public String toString() { return map.toString(); }
}

// --- TEST ---
Tokens ops = new Tokens();
ops.put('+', 4, (a, b) -> a + b);
ops.put('-', 4, (a, b) -> a - b);
ops.put('*', 3, (a, b) -> a * b);
ops.put('/', 3, (a, b) -> a / b);
ops.put('%', 3, (a, b) -> a % b);

System.out.println("Operator map: " + ops);
System.out.println("Contains '+'? " + ops.contains('+'));
System.out.println("Contains 'x'? " + ops.contains('x'));
System.out.println("Precedence of '+': " + ops.get('+').getPrecedence());
System.out.println("Precedence of '*': " + ops.get('*').getPrecedence());
System.out.println("10 / 3 = " + ops.get('/').calculate(10.0, 3.0));
System.out.println("10 % 3 = " + ops.get('%').calculate(10.0, 3.0));

// TODO: Add power operator (^, precedence 2, Math.pow) and test 2^8
{% endcapture %}

{% include code-runner.html
   runner_id="rpn_tokens"
   language="java"
   challenge=challenge_tokens
   code=code_tokens
   height="550px"
%}


<div class="rpn-popcorn">
    <div class="rpn-popcorn__header" style="background: linear-gradient(135deg, #e67e22 0%, #d35400 100%); border-bottom-color: #fff;">
        <h3 style="color: #fff;">Popcorn Hack 3: HashMap Lookup</h3>
        <p>The <code>Tokens</code> class uses <code>HashMap&lt;Character, TermOrOperator&gt;</code>. What is the time complexity of checking <code>operators.contains('+')</code>?</p>
    </div>
    <div class="rpn-popcorn__body" style="background: #2c3e50;">
        <div class="rpn-popcorn__options" id="ph3-options">
            <button onclick="checkPH3(0)" class="rpn-popcorn__option">A) O(n) — it scans through all entries</button>
            <button onclick="checkPH3(1)" class="rpn-popcorn__option">B) O(log n) — it uses binary search</button>
            <button onclick="checkPH3(2)" class="rpn-popcorn__option">C) O(1) — HashMap uses hashing for constant-time lookup</button>
            <button onclick="checkPH3(3)" class="rpn-popcorn__option">D) O(n squared) — it compares every pair</button>
        </div>
        <div id="ph3-feedback" class="rpn-popcorn__feedback"></div>
    </div>
</div>

<script>
function checkPH3(sel) {
    const btns = document.querySelectorAll('#ph3-options .rpn-popcorn__option');
    const fb = document.getElementById('ph3-feedback');
    btns.forEach(b => b.className = 'rpn-popcorn__option');
    if (sel === 2) {
        btns[2].classList.add('rpn-popcorn__option--correct');
        fb.className = 'rpn-popcorn__feedback rpn-popcorn__feedback--correct';
        fb.innerHTML = '<strong style="color:#2ecc71;">Correct!</strong> <code>HashMap.containsKey()</code> is <strong>O(1) average case</strong>. It computes the hash of the key character and jumps directly to the bucket. This is why we chose HashMap over ArrayList for operator lookup — when tokenizing long expressions, every character check is instant.';
    } else {
        btns[sel].classList.add('rpn-popcorn__option--incorrect');
        fb.className = 'rpn-popcorn__feedback rpn-popcorn__feedback--incorrect';
        if (sel === 0) fb.innerHTML = '<strong style="color:#e74c3c;">Incorrect.</strong> O(n) scanning is what ArrayList does. HashMap avoids this by using a hash function to jump directly to the right bucket.';
        if (sel === 1) fb.innerHTML = '<strong style="color:#e74c3c;">Incorrect.</strong> O(log n) binary search is what TreeMap uses (sorted keys). HashMap is even faster — O(1) via hashing.';
        if (sel === 3) fb.innerHTML = '<strong style="color:#e74c3c;">Incorrect.</strong> O(n squared) would be incredibly slow. HashMap is designed for fast lookups, not exhaustive comparisons.';
    }
}
</script>


---

## Step 4: Tokenizing an Expression

Before we can do any math, we need to **break the expression string into tokens**. This is called **tokenization** — the same concept compilers use to parse source code.

```
"(100 + 200) * 3" → [(, 100, +, 200, ), *, 3]
```

The algorithm walks character by character:
- If it's a **digit or decimal point** → accumulate into a multi-character number
- If it's an **operator or separator** → flush the accumulated number, then add the operator
- **Spaces** are separators but don't get added as tokens

### Challenge: Run the tokenizer on different expressions to see how it breaks them down.

In [ ]:
import java.util.*;
import java.util.function.BiFunction;

class Token {
    private final Character token; private final int precedence;
    private final BiFunction<Double, Double, Double> calc; private final int numArgs;
    public Token() { this('0'); }
    public Token(Character t) { this(t,0,null,0); }
    public Token(Character t, int p, BiFunction<Double,Double,Double> c, int n) { token=t; precedence=p; calc=c; numArgs=n; }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return calc.apply(a,b); }
    public String toString() { return token.toString(); }
}
class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String v) { value=v; }
    public TermOrOperator(Character t) { super(t); value=null; }
    public TermOrOperator(Character t, int p, BiFunction<Double,Double,Double> c) { super(t,p,c,2); value=null; }
    public TermOrOperator(Character t, int p, BiFunction<Double,Double,Double> c, int n) { super(t,p,c,n); value=null; }
    public String getValue() { return value; }
    public String toString() { return value!=null ? value : super.toString(); }
}
class Tokens {
    private Map<Character,TermOrOperator> map = new HashMap<>();
    public void put(Character t) { map.put(t, new TermOrOperator(t)); }
    public void put(Character t, int p, BiFunction<Double,Double,Double> c) { map.put(t, new TermOrOperator(t,p,c)); }
    public void put(Character t, int p, BiFunction<Double,Double,Double> c, int n) { map.put(t, new TermOrOperator(t,p,c,n)); }
    public TermOrOperator get(Character t) { return map.get(t); }
    public boolean contains(Character t) { return map.containsKey(t); }
}

// --- Tokenizer ---
Tokens operators = new Tokens();
operators.put('+', 4, (a,b) -> a+b);
operators.put('-', 4, (a,b) -> a-b);
operators.put('*', 3, (a,b) -> a*b);
operators.put('/', 3, (a,b) -> a/b);
operators.put('%', 3, (a,b) -> a%b);
operators.put('^', 2, (a,b) -> Math.pow(a,b));

Tokens separators = new Tokens();
separators.put(' ');
separators.put('(');
separators.put(')');

ArrayList<TermOrOperator> tokenize(String expr, Tokens ops, Tokens seps) {
    ArrayList<TermOrOperator> terms = new ArrayList<>();
    int start = 0;
    StringBuilder multi = new StringBuilder();
    for (int i = 0; i < expr.length(); i++) {
        char c = expr.charAt(i);
        if (ops.contains(c) || seps.contains(c)) {
            if (multi.length() > 0) {
                terms.add(new TermOrOperator(expr.substring(start, i)));
            }
            TermOrOperator t = ops.get(c);
            if (t == null) t = seps.get(c);
            if (t != null && t.getToken() != ' ') terms.add(t);
            start = i + 1;
            multi = new StringBuilder();
        } else {
            multi.append(c);
        }
    }
    if (multi.length() > 0) terms.add(new TermOrOperator(expr.substring(start)));
    return terms;
}

// --- TEST ---
String[] expressions = {
    "100 + 200 * 3",
    "(100 + 200) * 3",
    "3.14 * 2",
    "2 ^ 10",
    "100 % 7"
};
for (String expr : expressions) {
    System.out.println("\"" + expr + "\"  →  " + tokenize(expr, operators, separators));
}

// TODO: Try tokenizing your own expression!


{% capture challenge_tokenize %}
The tokenizer splits a math expression string into individual tokens. Run it to see how different expressions get tokenized. Try adding your own expression at the bottom.
{% endcapture %}

{% capture code_tokenize %}
import java.util.*;
import java.util.function.BiFunction;

class Token {
    private final Character token; private final int precedence;
    private final BiFunction<Double, Double, Double> calc; private final int numArgs;
    public Token() { this('0'); }
    public Token(Character t) { this(t,0,null,0); }
    public Token(Character t, int p, BiFunction<Double,Double,Double> c, int n) { token=t; precedence=p; calc=c; numArgs=n; }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return calc.apply(a,b); }
    public String toString() { return token.toString(); }
}
class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String v) { value=v; }
    public TermOrOperator(Character t) { super(t); value=null; }
    public TermOrOperator(Character t, int p, BiFunction<Double,Double,Double> c) { super(t,p,c,2); value=null; }
    public TermOrOperator(Character t, int p, BiFunction<Double,Double,Double> c, int n) { super(t,p,c,n); value=null; }
    public String getValue() { return value; }
    public String toString() { return value!=null ? value : super.toString(); }
}
class Tokens {
    private Map<Character,TermOrOperator> map = new HashMap<>();
    public void put(Character t) { map.put(t, new TermOrOperator(t)); }
    public void put(Character t, int p, BiFunction<Double,Double,Double> c) { map.put(t, new TermOrOperator(t,p,c)); }
    public void put(Character t, int p, BiFunction<Double,Double,Double> c, int n) { map.put(t, new TermOrOperator(t,p,c,n)); }
    public TermOrOperator get(Character t) { return map.get(t); }
    public boolean contains(Character t) { return map.containsKey(t); }
}

// --- Tokenizer ---
Tokens operators = new Tokens();
operators.put('+', 4, (a,b) -> a+b);
operators.put('-', 4, (a,b) -> a-b);
operators.put('*', 3, (a,b) -> a*b);
operators.put('/', 3, (a,b) -> a/b);
operators.put('%', 3, (a,b) -> a%b);
operators.put('^', 2, (a,b) -> Math.pow(a,b));

Tokens separators = new Tokens();
separators.put(' ');
separators.put('(');
separators.put(')');

ArrayList<TermOrOperator> tokenize(String expr, Tokens ops, Tokens seps) {
    ArrayList<TermOrOperator> terms = new ArrayList<>();
    int start = 0;
    StringBuilder multi = new StringBuilder();
    for (int i = 0; i < expr.length(); i++) {
        char c = expr.charAt(i);
        if (ops.contains(c) || seps.contains(c)) {
            if (multi.length() > 0) {
                terms.add(new TermOrOperator(expr.substring(start, i)));
            }
            TermOrOperator t = ops.get(c);
            if (t == null) t = seps.get(c);
            if (t != null && t.getToken() != ' ') terms.add(t);
            start = i + 1;
            multi = new StringBuilder();
        } else {
            multi.append(c);
        }
    }
    if (multi.length() > 0) terms.add(new TermOrOperator(expr.substring(start)));
    return terms;
}

// --- TEST ---
String[] expressions = {
    "100 + 200 * 3",
    "(100 + 200) * 3",
    "3.14 * 2",
    "2 ^ 10",
    "100 % 7"
};
for (String expr : expressions) {
    System.out.println("\"" + expr + "\"  →  " + tokenize(expr, operators, separators));
}

// TODO: Try tokenizing your own expression!
{% endcapture %}

{% include code-runner.html
   runner_id="rpn_tokenize"
   language="java"
   challenge=challenge_tokenize
   code=code_tokenize
   height="550px"
%}


<div class="rpn-reveal">
    <div class="rpn-reveal__header">
        <h3>Trace It: How Does Tokenization Work?</h3>
    </div>
    <div class="rpn-reveal__body">
        <div class="rpn-reveal__code-block">
            <pre><code>"(100 + 200) * 3"</code></pre>
            <p style="color:#888; margin-top:10px; font-size:13px;">Walk through each character. What happens at each step?</p>
        </div>
        <button onclick="toggleTokenTrace()" id="tokenTraceBtn" class="rpn-reveal__toggle-btn">Show Step-by-Step Trace</button>
        <div id="tokenTraceAnswer" class="rpn-reveal__answer">
            <h4>Character-by-Character Trace:</h4>
            <div class="rpn-reveal__step">
                <div class="step-label">Step 1: char = '('</div>
                <div class="step-detail">Separator found — <code>multi</code> is empty, skip number flush — Add <code>(</code> token — Result: <code>[(]</code></div>
            </div>
            <div class="rpn-reveal__step">
                <div class="step-label">Step 2-4: chars = '1', '0', '0'</div>
                <div class="step-detail">Not an operator or separator — Accumulate into <code>multi = "100"</code> — No token added yet</div>
            </div>
            <div class="rpn-reveal__step">
                <div class="step-label">Step 5: char = ' '</div>
                <div class="step-detail">Separator found — Flush <code>multi</code> — Add number token <code>"100"</code> — Space is not added (filtered out) — Result: <code>[(, 100]</code></div>
            </div>
            <div class="rpn-reveal__step">
                <div class="step-label">Step 6: char = '+'</div>
                <div class="step-detail">Operator found — <code>multi</code> is empty — Add <code>+</code> operator token — Result: <code>[(, 100, +]</code></div>
            </div>
            <div class="rpn-reveal__step">
                <div class="step-label">Steps 7-10: ' ', '2', '0', '0'</div>
                <div class="step-detail">Space flushes nothing (multi empty), then accumulate <code>"200"</code></div>
            </div>
            <div class="rpn-reveal__step">
                <div class="step-label">Step 11: char = ')'</div>
                <div class="step-detail">Separator found — Flush <code>"200"</code> — Add <code>)</code> token — Result: <code>[(, 100, +, 200, )]</code></div>
            </div>
            <div class="rpn-reveal__step">
                <div class="step-label">Steps 12-15: ' ', '*', ' ', '3'</div>
                <div class="step-detail">Space, then <code>*</code> operator added, then space, then <code>"3"</code> accumulated and flushed at end-of-string</div>
            </div>
            <div class="rpn-reveal__key-point">
                <div class="key-title">Final Result</div>
                <div class="key-detail"><code>[(, 100, +, 200, ), *, 3]</code> — 7 tokens from a 15-character string. Notice how spaces are used as delimiters but never become tokens themselves.</div>
            </div>
        </div>
    </div>
</div>

<script>
function toggleTokenTrace() {
    const ans = document.getElementById('tokenTraceAnswer');
    const btn = document.getElementById('tokenTraceBtn');
    if (ans.style.display === 'none' || ans.style.display === '') {
        ans.style.display = 'block';
        btn.textContent = 'Hide Trace';
        btn.style.background = '#e74c3c';
    } else {
        ans.style.display = 'none';
        btn.textContent = 'Show Step-by-Step Trace';
        btn.style.background = '#4ecdc4';
    }
}
</script>


---

## Step 5: Converting to Reverse Polish Notation (The Stack Algorithm)

This is the **core algorithm**. RPN reorders tokens so that operators come AFTER their operands, eliminating the need for parentheses.

**Infix (normal):** `(100 + 200) * 3`  
**RPN (postfix):** `100 200 + 3 *`

### The Shunting-Yard Algorithm (by Dijkstra)

Uses a **Stack** to reorder operators:

1. **Number** → goes straight to output
2. **`(`** → push onto operator stack
3. **`)`** → pop operators to output until `(` is found
4. **Operator** → pop higher-precedence operators from stack to output, then push current
5. **End** → pop remaining operators to output

### Challenge: Trace the algorithm by hand for `(100 + 200) * 3`, then run to verify.

In [ ]:
import java.util.*;
import java.util.function.BiFunction;

class Token {
    private final Character token; private final int precedence;
    private final BiFunction<Double,Double,Double> calc; private final int numArgs;
    public Token() { this('0'); }
    public Token(Character t) { this(t,0,null,0); }
    public Token(Character t,int p,BiFunction<Double,Double,Double> c,int n) { token=t;precedence=p;calc=c;numArgs=n; }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return precedence>t.getPrecedence(); }
    public Double calculate(Double a,Double b) { return calc.apply(a,b); }
    public String toString() { return token.toString(); }
}
class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String v) { value=v; }
    public TermOrOperator(Character t) { super(t); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c) { super(t,p,c,2); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c,int n) { super(t,p,c,n); value=null; }
    public String getValue() { return value; }
    public String toString() { return value!=null?value:super.toString(); }
}
class Tokens {
    Map<Character,TermOrOperator> map=new HashMap<>();
    void put(Character t) { map.put(t,new TermOrOperator(t)); }
    void put(Character t,int p,BiFunction<Double,Double,Double> c) { map.put(t,new TermOrOperator(t,p,c)); }
    TermOrOperator get(Character t) { return map.get(t); }
    boolean contains(Character t) { return map.containsKey(t); }
}

Tokens ops = new Tokens();
ops.put('+',4,(a,b)->a+b); ops.put('-',4,(a,b)->a-b);
ops.put('*',3,(a,b)->a*b); ops.put('/',3,(a,b)->a/b);
ops.put('^',2,(a,b)->Math.pow(a,b));
Tokens seps = new Tokens();
seps.put(' '); seps.put('('); seps.put(')');

ArrayList<TermOrOperator> tokenize(String expr) {
    ArrayList<TermOrOperator> terms = new ArrayList<>();
    int start=0; StringBuilder multi=new StringBuilder();
    for (int i=0;i<expr.length();i++) {
        char c=expr.charAt(i);
        if (ops.contains(c)||seps.contains(c)) {
            if (multi.length()>0) terms.add(new TermOrOperator(expr.substring(start,i)));
            TermOrOperator t=ops.get(c); if(t==null) t=seps.get(c);
            if (t!=null && t.getToken()!=' ') terms.add(t);
            start=i+1; multi=new StringBuilder();
        } else multi.append(c);
    }
    if (multi.length()>0) terms.add(new TermOrOperator(expr.substring(start)));
    return terms;
}

// --- Shunting-Yard: Infix → RPN ---
ArrayList<TermOrOperator> toRPN(ArrayList<TermOrOperator> terms) {
    ArrayList<TermOrOperator> rpn = new ArrayList<>();
    Stack<TermOrOperator> opStack = new Stack<>();
    for (TermOrOperator term : terms) {
        if (term.getToken() == '(') {
            opStack.push(term);
        } else if (term.getToken() == ')') {
            while (opStack.peek().getToken() != '(') rpn.add(opStack.pop());
            opStack.pop(); // remove '('
        } else if (ops.contains(term.getToken())) {
            while (!opStack.isEmpty() && ops.contains(opStack.peek().getToken()) && term.isPrecedent(opStack.peek()))
                rpn.add(opStack.pop());
            opStack.push(term);
        } else {
            rpn.add(term);
        }
    }
    while (!opStack.isEmpty()) rpn.add(opStack.pop());
    return rpn;
}

// --- TEST ---
String[] expressions = {
    "100 + 200 * 3",
    "(100 + 200) * 3",
    "3 + 4 * 2 - 1",
    "2 ^ 10"
};
for (String expr : expressions) {
    ArrayList<TermOrOperator> tokens = tokenize(expr);
    ArrayList<TermOrOperator> rpn = toRPN(tokens);
    System.out.println("Infix:  " + expr);
    System.out.println("Tokens: " + tokens);
    System.out.println("RPN:    " + rpn);
    System.out.println();
}


{% capture challenge_rpn %}
The Shunting-Yard algorithm converts infix to RPN using a Stack. Run to see step-by-step conversion. Try tracing "5 + 3 * 2" by hand first, then verify with the code.
{% endcapture %}

{% capture code_rpn %}
import java.util.*;
import java.util.function.BiFunction;

class Token {
    private final Character token; private final int precedence;
    private final BiFunction<Double,Double,Double> calc; private final int numArgs;
    public Token() { this('0'); }
    public Token(Character t) { this(t,0,null,0); }
    public Token(Character t,int p,BiFunction<Double,Double,Double> c,int n) { token=t;precedence=p;calc=c;numArgs=n; }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return precedence>t.getPrecedence(); }
    public Double calculate(Double a,Double b) { return calc.apply(a,b); }
    public String toString() { return token.toString(); }
}
class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String v) { value=v; }
    public TermOrOperator(Character t) { super(t); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c) { super(t,p,c,2); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c,int n) { super(t,p,c,n); value=null; }
    public String getValue() { return value; }
    public String toString() { return value!=null?value:super.toString(); }
}
class Tokens {
    Map<Character,TermOrOperator> map=new HashMap<>();
    void put(Character t) { map.put(t,new TermOrOperator(t)); }
    void put(Character t,int p,BiFunction<Double,Double,Double> c) { map.put(t,new TermOrOperator(t,p,c)); }
    TermOrOperator get(Character t) { return map.get(t); }
    boolean contains(Character t) { return map.containsKey(t); }
}

Tokens ops = new Tokens();
ops.put('+',4,(a,b)->a+b); ops.put('-',4,(a,b)->a-b);
ops.put('*',3,(a,b)->a*b); ops.put('/',3,(a,b)->a/b);
ops.put('^',2,(a,b)->Math.pow(a,b));
Tokens seps = new Tokens();
seps.put(' '); seps.put('('); seps.put(')');

ArrayList<TermOrOperator> tokenize(String expr) {
    ArrayList<TermOrOperator> terms = new ArrayList<>();
    int start=0; StringBuilder multi=new StringBuilder();
    for (int i=0;i<expr.length();i++) {
        char c=expr.charAt(i);
        if (ops.contains(c)||seps.contains(c)) {
            if (multi.length()>0) terms.add(new TermOrOperator(expr.substring(start,i)));
            TermOrOperator t=ops.get(c); if(t==null) t=seps.get(c);
            if (t!=null && t.getToken()!=' ') terms.add(t);
            start=i+1; multi=new StringBuilder();
        } else multi.append(c);
    }
    if (multi.length()>0) terms.add(new TermOrOperator(expr.substring(start)));
    return terms;
}

// --- Shunting-Yard: Infix to RPN ---
ArrayList<TermOrOperator> toRPN(ArrayList<TermOrOperator> terms) {
    ArrayList<TermOrOperator> rpn = new ArrayList<>();
    Stack<TermOrOperator> opStack = new Stack<>();
    for (TermOrOperator term : terms) {
        if (term.getToken() == '(') {
            opStack.push(term);
        } else if (term.getToken() == ')') {
            while (opStack.peek().getToken() != '(') rpn.add(opStack.pop());
            opStack.pop(); // remove '('
        } else if (ops.contains(term.getToken())) {
            while (!opStack.isEmpty() && ops.contains(opStack.peek().getToken()) && term.isPrecedent(opStack.peek()))
                rpn.add(opStack.pop());
            opStack.push(term);
        } else {
            rpn.add(term);
        }
    }
    while (!opStack.isEmpty()) rpn.add(opStack.pop());
    return rpn;
}

// --- TEST ---
String[] expressions = {
    "100 + 200 * 3",
    "(100 + 200) * 3",
    "3 + 4 * 2 - 1",
    "2 ^ 10"
};
for (String expr : expressions) {
    ArrayList<TermOrOperator> tokens = tokenize(expr);
    ArrayList<TermOrOperator> rpn = toRPN(tokens);
    System.out.println("Infix:  " + expr);
    System.out.println("Tokens: " + tokens);
    System.out.println("RPN:    " + rpn);
    System.out.println();
}
{% endcapture %}

{% include code-runner.html
   runner_id="rpn_convert"
   language="java"
   challenge=challenge_rpn
   code=code_rpn
   height="550px"
%}


<div class="rpn-dragdrop">
    <div class="rpn-dragdrop__inner">
        <h3 class="rpn-dragdrop__title">RPN Ordering Challenge</h3>
        <p class="rpn-dragdrop__subtitle">Drag the tokens into correct Reverse Polish Notation order for: <strong style="color:#667eea;">(5 + 3) * 2</strong></p>

        <div class="rpn-dragdrop__area">
            <div class="rpn-dragdrop__column">
                <h4>Available Tokens</h4>
                <div class="rpn-dragdrop__items" id="rpnDragSource"></div>
            </div>
            <div class="rpn-dragdrop__column">
                <h4>Your RPN Order</h4>
                <div id="rpnDropTarget"></div>
            </div>
        </div>

        <div class="rpn-dragdrop__controls">
            <button class="rpn-dragdrop__btn rpn-dragdrop__btn--check" onclick="checkRPNOrder()">Check Answer</button>
            <button class="rpn-dragdrop__btn rpn-dragdrop__btn--clear" onclick="clearRPNOrder()">Clear All</button>
            <button class="rpn-dragdrop__btn rpn-dragdrop__btn--reset" onclick="resetRPNGame()">Reset & Shuffle</button>
        </div>

        <div id="rpnDragFeedback" class="rpn-dragdrop__feedback"></div>
    </div>
</div>

<script>
(function() {
    const correctOrder = ['5', '3', '+', '2', '*'];
    const tokens = ['*', '5', '+', '3', '2'];
    let currentTokens = [];
    let draggedEl = null;

    function shuffle(arr) {
        const a = [...arr];
        for (let i = a.length - 1; i > 0; i--) {
            const j = Math.floor(Math.random() * (i + 1));
            [a[i], a[j]] = [a[j], a[i]];
        }
        return a;
    }

    function initRPN() {
        currentTokens = shuffle(tokens);
        renderSource();
        renderTarget();
        document.getElementById('rpnDragFeedback').className = 'rpn-dragdrop__feedback';
    }

    function renderSource() {
        const src = document.getElementById('rpnDragSource');
        src.innerHTML = '';
        currentTokens.forEach((t, i) => {
            const el = document.createElement('div');
            el.className = 'rpn-dragdrop__item';
            el.draggable = true;
            el.textContent = t;
            el.dataset.idx = i;
            el.addEventListener('dragstart', e => { draggedEl = e.target; e.target.classList.add('dragging'); });
            el.addEventListener('dragend', e => e.target.classList.remove('dragging'));
            src.appendChild(el);
        });
    }

    function renderTarget() {
        const tgt = document.getElementById('rpnDropTarget');
        tgt.innerHTML = '';
        for (let i = 0; i < 5; i++) {
            const slot = document.createElement('div');
            slot.className = 'rpn-dragdrop__slot';
            const label = document.createElement('div');
            label.className = 'slot-label';
            label.textContent = 'Position ' + (i + 1);
            const dz = document.createElement('div');
            dz.className = 'rpn-dragdrop__dropzone';
            dz.dataset.pos = i;
            dz.innerHTML = '<span class="placeholder">Drop here</span>';
            dz.addEventListener('dragover', e => { e.preventDefault(); dz.classList.add('drag-over'); });
            dz.addEventListener('dragleave', () => dz.classList.remove('drag-over'));
            dz.addEventListener('drop', e => {
                e.preventDefault();
                dz.classList.remove('drag-over');
                if (draggedEl) {
                    dz.innerHTML = '';
                    dz.textContent = draggedEl.textContent;
                    dz.classList.add('filled');
                    dz.dataset.token = draggedEl.textContent;
                    draggedEl.remove();
                }
            });
            slot.appendChild(label);
            slot.appendChild(dz);
            tgt.appendChild(slot);
        }
    }

    window.checkRPNOrder = function() {
        const zones = document.querySelectorAll('#rpnDropTarget .rpn-dragdrop__dropzone');
        const fb = document.getElementById('rpnDragFeedback');
        const userOrder = [];
        let allFilled = true;
        zones.forEach(z => {
            if (!z.dataset.token) allFilled = false;
            else userOrder.push(z.dataset.token);
        });
        if (!allFilled) {
            fb.className = 'rpn-dragdrop__feedback rpn-dragdrop__feedback--warning';
            fb.innerHTML = '<strong>Incomplete!</strong> Fill all 5 positions before checking.';
            return;
        }
        const correct = userOrder.every((t, i) => t === correctOrder[i]);
        if (correct) {
            fb.className = 'rpn-dragdrop__feedback rpn-dragdrop__feedback--success';
            fb.innerHTML = '<strong>Perfect!</strong> <code>5 3 + 2 *</code> is correct RPN for <code>(5 + 3) * 2</code>.<br>The stack processes: push 5, push 3, pop both & add — 8, push 2, pop both & multiply — <strong>16</strong>.';
        } else {
            fb.className = 'rpn-dragdrop__feedback rpn-dragdrop__feedback--error';
            fb.innerHTML = '<strong>Not quite.</strong> Remember: in RPN, operands come before their operator. Parenthesized <code>(5 + 3)</code> means 5 and 3 must appear before <code>+</code>.';
        }
    };

    window.clearRPNOrder = function() {
        initRPN();
    };

    window.resetRPNGame = function() {
        initRPN();
    };

    initRPN();
})();
</script>


---

## Step 6: Evaluating RPN — The Final Calculation

Once we have RPN, evaluation is beautifully simple using a **Stack**:

1. Read tokens left to right
2. **Number** → push onto stack
3. **Operator** → pop operand(s), calculate, push result back
4. **Done** → one number remains on stack = final answer

### Example: `100 200 + 3 *`

| Step | Token | Stack | Action |
|------|-------|-------|--------|
| 1 | 100 | [100] | Push |
| 2 | 200 | [100, 200] | Push |
| 3 | + | [300] | Pop 200 & 100, add, push 300 |
| 4 | 3 | [300, 3] | Push |
| 5 | * | [900] | Pop 3 & 300, multiply, push 900 |

**Result: 900**


<div class="rpn-stack-viz">
    <div class="rpn-stack-viz__header">
        <h3>Interactive Stack Visualization</h3>
        <p>Step through RPN evaluation of <code>100 200 + 3 *</code> and watch the stack in action.</p>
    </div>
    <div class="rpn-stack-viz__body">
        <div class="rpn-stack-viz__controls">
            <button class="rpn-stack-viz__step-btn" id="stackStepBtn" onclick="stackStep()">Next Step</button>
            <button class="rpn-stack-viz__step-btn rpn-stack-viz__step-btn--reset" onclick="stackReset()">Reset</button>
        </div>
        <div class="rpn-stack-viz__display">
            <div class="rpn-stack-viz__token-stream">
                <h5>Token Stream</h5>
                <div id="stackTokens"></div>
            </div>
            <div class="rpn-stack-viz__stack-container">
                <h5>Stack</h5>
                <div class="rpn-stack-viz__stack" id="stackDisplay"></div>
            </div>
        </div>
        <div class="rpn-stack-viz__narration" id="stackNarration">Click <strong>Next Step</strong> to begin evaluation.</div>
    </div>
</div>

<script>
(function() {
    const rpnTokens = ['100', '200', '+', '3', '*'];
    const steps = [
        { active: 0, stack: ['100'], narration: '<strong>Push 100</strong> — it\'s a number, so push it onto the stack.' },
        { active: 1, stack: ['100', '200'], narration: '<strong>Push 200</strong> — another number, push it.' },
        { active: 2, stack: ['300'], narration: '<strong>Operator +</strong> — pop 200 and 100, calculate 100 + 200 = 300, push result.' },
        { active: 3, stack: ['300', '3'], narration: '<strong>Push 3</strong> — number goes on the stack.' },
        { active: 4, stack: ['900'], narration: '<strong>Operator *</strong> — pop 3 and 300, calculate 300 x 3 = 900, push result.' },
        { active: -1, stack: ['900'], narration: '<strong>Done!</strong> One value remains on the stack: <strong>900</strong> — that\'s our answer!' }
    ];
    let currentStep = -1;

    function renderTokenStream() {
        const container = document.getElementById('stackTokens');
        container.innerHTML = '';
        rpnTokens.forEach((t, i) => {
            const el = document.createElement('span');
            el.className = 'rpn-stack-viz__token';
            if (currentStep >= 0 && i < steps[currentStep].active) el.classList.add('rpn-stack-viz__token--processed');
            if (currentStep >= 0 && i === steps[currentStep].active) el.classList.add('rpn-stack-viz__token--active');
            el.textContent = t;
            container.appendChild(el);
        });
    }

    function renderStack() {
        const container = document.getElementById('stackDisplay');
        container.innerHTML = '';
        if (currentStep >= 0) {
            steps[currentStep].stack.forEach(val => {
                const el = document.createElement('div');
                el.className = 'rpn-stack-viz__stack-item';
                el.textContent = val;
                container.appendChild(el);
            });
        }
    }

    function updateNarration() {
        const el = document.getElementById('stackNarration');
        if (currentStep < 0) el.innerHTML = 'Click <strong>Next Step</strong> to begin evaluation.';
        else el.innerHTML = steps[currentStep].narration;
    }

    window.stackStep = function() {
        if (currentStep < steps.length - 1) {
            currentStep++;
            renderTokenStream();
            renderStack();
            updateNarration();
        }
        if (currentStep >= steps.length - 1) {
            document.getElementById('stackStepBtn').disabled = true;
        }
    };

    window.stackReset = function() {
        currentStep = -1;
        document.getElementById('stackStepBtn').disabled = false;
        renderTokenStream();
        renderStack();
        updateNarration();
    };

    renderTokenStream();
    renderStack();
    updateNarration();
})();
</script>


---

## Step 7: The Complete Calculator

Now let's put it all together. The full `Calculator` class chains the entire pipeline:

```
Expression → Tokenize → RPN → Evaluate → Result
```

### Challenge: Run the complete calculator, then try your own expressions!

Supported operators: `+`, `-`, `*`, `/`, `%`, `^` (power), `√` (square root)

In [ ]:
import java.util.*;
import java.util.function.BiFunction;

class Token {
    private final Character token; private final int precedence;
    private final BiFunction<Double,Double,Double> calc; private final int numArgs;
    public Token() { this('0'); }
    public Token(Character t) { this(t,0,null,0); }
    public Token(Character t,int p,BiFunction<Double,Double,Double> c,int n) {
        token=t; precedence=p; calc=c; numArgs=n;
    }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return calc.apply(a,b); }
    public String toString() { return token.toString(); }
}

class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String v) { value=v; }
    public TermOrOperator(Character t) { super(t); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c) { super(t,p,c,2); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c,int n) { super(t,p,c,n); value=null; }
    public String getValue() { return value; }
    public String toString() { return value!=null ? value : super.toString(); }
}

class Tokens {
    private Map<Character,TermOrOperator> map = new HashMap<>();
    public void put(Character t) { map.put(t, new TermOrOperator(t)); }
    public void put(Character t,int p,BiFunction<Double,Double,Double> c) { map.put(t, new TermOrOperator(t,p,c)); }
    public void put(Character t,int p,BiFunction<Double,Double,Double> c,int n) { map.put(t, new TermOrOperator(t,p,c,n)); }
    public TermOrOperator get(Character t) { return map.get(t); }
    public int getPrecedence(Character t) { return map.get(t).getPrecedence(); }
    public boolean contains(Character t) { return map.containsKey(t); }
}

public class Calculator {
    private final String expression;
    private ArrayList<TermOrOperator> terms = new ArrayList<>();
    private ArrayList<TermOrOperator> rpnTerms = new ArrayList<>();
    private Tokens operators = new Tokens();
    private Tokens separators = new Tokens();
    private Double result = 0.0;

    public Calculator(String expression) {
        initOperators();
        initSeparators();
        this.expression = expression;
        this.termTokenizer();
        this.termsToRPN();
        this.rpnToResult();
    }

    private void initOperators() {
        operators.put('*', 3, (a,b) -> a*b);
        operators.put('/', 3, (a,b) -> a/b);
        operators.put('%', 3, (a,b) -> a%b);
        operators.put('+', 4, (a,b) -> a+b);
        operators.put('-', 4, (a,b) -> a-b);
        operators.put('^', 2, (a,b) -> Math.pow(a,b));
    }

    private void initSeparators() {
        separators.put(' ');
        separators.put('(');
        separators.put(')');
    }

    private void termTokenizer() {
        int start = 0;
        StringBuilder multi = new StringBuilder();
        for (int i = 0; i < expression.length(); i++) {
            char c = expression.charAt(i);
            if (operators.contains(c) || separators.contains(c)) {
                if (multi.length() > 0)
                    terms.add(new TermOrOperator(expression.substring(start, i)));
                TermOrOperator t = operators.get(c);
                if (t == null) t = separators.get(c);
                if (t != null && t.getToken() != ' ') terms.add(t);
                start = i + 1;
                multi = new StringBuilder();
            } else {
                multi.append(c);
            }
        }
        if (multi.length() > 0) terms.add(new TermOrOperator(expression.substring(start)));
    }

    private void termsToRPN() {
        Stack<TermOrOperator> opStack = new Stack<>();
        for (TermOrOperator term : terms) {
            if (term.getToken() == '(') {
                opStack.push(term);
            } else if (term.getToken() == ')') {
                while (opStack.peek().getToken() != '(') rpnTerms.add(opStack.pop());
                opStack.pop();
            } else if (operators.contains(term.getToken())) {
                while (!opStack.isEmpty() && operators.contains(opStack.peek().getToken())
                       && term.isPrecedent(opStack.peek()))
                    rpnTerms.add(opStack.pop());
                opStack.push(term);
            } else {
                rpnTerms.add(term);
            }
        }
        while (!opStack.isEmpty()) rpnTerms.add(opStack.pop());
    }

    private void rpnToResult() {
        Stack<Double> calcStack = new Stack<>();
        for (TermOrOperator term : rpnTerms) {
            if (operators.contains(term.getToken())) {
                Double op2 = calcStack.pop();
                Double op1 = calcStack.pop();
                calcStack.push(term.calculate(op1, op2));
            } else {
                calcStack.push(Double.valueOf(term.getValue()));
            }
        }
        this.result = calcStack.pop();
    }

    public String toString() {
        return "Original expression: " + expression + "\n"
             + "Tokenized expression: " + terms + "\n"
             + "Reverse Polish Notation: " + rpnTerms + "\n"
             + "Final result: " + String.format("%.2f", result);
    }

    public static void main(String[] args) {
        System.out.println("=== Simple Math ===");
        System.out.println(new Calculator("100 + 200 * 3"));
        System.out.println();

        System.out.println("=== Parentheses Override Precedence ===");
        System.out.println(new Calculator("(100 + 200) * 3"));
        System.out.println();

        System.out.println("=== Decimals ===");
        System.out.println(new Calculator("100.2 - 99.3"));
        System.out.println();

        System.out.println("=== Modulo ===");
        System.out.println(new Calculator("300 % 200"));
        System.out.println();

        System.out.println("=== Power ===");
        System.out.println(new Calculator("2 ^ 10"));
        System.out.println();

        // TODO: Add your own expression here!
        // System.out.println(new Calculator("YOUR EXPRESSION"));
    }
}
Calculator.main(null);


{% capture challenge_full %}
The complete RPN Calculator! Run it to see the full pipeline — from expression to tokenization to RPN to result. Try adding your own expressions at the bottom of main().
{% endcapture %}

{% capture code_full %}
import java.util.*;
import java.util.function.BiFunction;

class Token {
    private final Character token; private final int precedence;
    private final BiFunction<Double,Double,Double> calc; private final int numArgs;
    public Token() { this('0'); }
    public Token(Character t) { this(t,0,null,0); }
    public Token(Character t,int p,BiFunction<Double,Double,Double> c,int n) {
        token=t; precedence=p; calc=c; numArgs=n;
    }
    public Character getToken() { return token; }
    public int getPrecedence() { return precedence; }
    public int getNumArgs() { return numArgs; }
    public boolean isPrecedent(Token t) { return precedence > t.getPrecedence(); }
    public Double calculate(Double a, Double b) { return calc.apply(a,b); }
    public String toString() { return token.toString(); }
}

class TermOrOperator extends Token {
    private final String value;
    public TermOrOperator(String v) { value=v; }
    public TermOrOperator(Character t) { super(t); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c) { super(t,p,c,2); value=null; }
    public TermOrOperator(Character t,int p,BiFunction<Double,Double,Double> c,int n) { super(t,p,c,n); value=null; }
    public String getValue() { return value; }
    public String toString() { return value!=null ? value : super.toString(); }
}

class Tokens {
    private Map<Character,TermOrOperator> map = new HashMap<>();
    public void put(Character t) { map.put(t, new TermOrOperator(t)); }
    public void put(Character t,int p,BiFunction<Double,Double,Double> c) { map.put(t, new TermOrOperator(t,p,c)); }
    public void put(Character t,int p,BiFunction<Double,Double,Double> c,int n) { map.put(t, new TermOrOperator(t,p,c,n)); }
    public TermOrOperator get(Character t) { return map.get(t); }
    public int getPrecedence(Character t) { return map.get(t).getPrecedence(); }
    public boolean contains(Character t) { return map.containsKey(t); }
}

public class Calculator {
    private final String expression;
    private ArrayList<TermOrOperator> terms = new ArrayList<>();
    private ArrayList<TermOrOperator> rpnTerms = new ArrayList<>();
    private Tokens operators = new Tokens();
    private Tokens separators = new Tokens();
    private Double result = 0.0;

    public Calculator(String expression) {
        initOperators();
        initSeparators();
        this.expression = expression;
        this.termTokenizer();
        this.termsToRPN();
        this.rpnToResult();
    }

    private void initOperators() {
        operators.put('*', 3, (a,b) -> a*b);
        operators.put('/', 3, (a,b) -> a/b);
        operators.put('%', 3, (a,b) -> a%b);
        operators.put('+', 4, (a,b) -> a+b);
        operators.put('-', 4, (a,b) -> a-b);
        operators.put('^', 2, (a,b) -> Math.pow(a,b));
    }

    private void initSeparators() {
        separators.put(' ');
        separators.put('(');
        separators.put(')');
    }

    private void termTokenizer() {
        int start = 0;
        StringBuilder multi = new StringBuilder();
        for (int i = 0; i < expression.length(); i++) {
            char c = expression.charAt(i);
            if (operators.contains(c) || separators.contains(c)) {
                if (multi.length() > 0)
                    terms.add(new TermOrOperator(expression.substring(start, i)));
                TermOrOperator t = operators.get(c);
                if (t == null) t = separators.get(c);
                if (t != null && t.getToken() != ' ') terms.add(t);
                start = i + 1;
                multi = new StringBuilder();
            } else {
                multi.append(c);
            }
        }
        if (multi.length() > 0) terms.add(new TermOrOperator(expression.substring(start)));
    }

    private void termsToRPN() {
        Stack<TermOrOperator> opStack = new Stack<>();
        for (TermOrOperator term : terms) {
            if (term.getToken() == '(') {
                opStack.push(term);
            } else if (term.getToken() == ')') {
                while (opStack.peek().getToken() != '(') rpnTerms.add(opStack.pop());
                opStack.pop();
            } else if (operators.contains(term.getToken())) {
                while (!opStack.isEmpty() && operators.contains(opStack.peek().getToken())
                       && term.isPrecedent(opStack.peek()))
                    rpnTerms.add(opStack.pop());
                opStack.push(term);
            } else {
                rpnTerms.add(term);
            }
        }
        while (!opStack.isEmpty()) rpnTerms.add(opStack.pop());
    }

    private void rpnToResult() {
        Stack<Double> calcStack = new Stack<>();
        for (TermOrOperator term : rpnTerms) {
            if (operators.contains(term.getToken())) {
                Double op2 = calcStack.pop();
                Double op1 = calcStack.pop();
                calcStack.push(term.calculate(op1, op2));
            } else {
                calcStack.push(Double.valueOf(term.getValue()));
            }
        }
        this.result = calcStack.pop();
    }

    public String toString() {
        return "Original expression: " + expression + "\n"
             + "Tokenized expression: " + terms + "\n"
             + "Reverse Polish Notation: " + rpnTerms + "\n"
             + "Final result: " + String.format("%.2f", result);
    }

    public static void main(String[] args) {
        System.out.println("=== Simple Math ===");
        System.out.println(new Calculator("100 + 200 * 3"));
        System.out.println();

        System.out.println("=== Parentheses Override Precedence ===");
        System.out.println(new Calculator("(100 + 200) * 3"));
        System.out.println();

        System.out.println("=== Decimals ===");
        System.out.println(new Calculator("100.2 - 99.3"));
        System.out.println();

        System.out.println("=== Modulo ===");
        System.out.println(new Calculator("300 % 200"));
        System.out.println();

        System.out.println("=== Power ===");
        System.out.println(new Calculator("2 ^ 10"));
        System.out.println();

        // TODO: Add your own expression here!
        // System.out.println(new Calculator("YOUR EXPRESSION"));
    }
}
Calculator.main(null);
{% endcapture %}

{% include code-runner.html
   runner_id="rpn_full"
   language="java"
   challenge=challenge_full
   code=code_full
   height="600px"
%}


---

## Knowledge Check

Test your understanding of the full RPN Calculator pipeline!

<div class="rpn-quiz">
    <div class="rpn-quiz__header">
        <h3>Final Knowledge Check</h3>
        <p>5 questions covering the entire lesson</p>
    </div>
    <div class="rpn-quiz__body">
        <!-- Q1 -->
        <div class="rpn-quiz__question" id="kq1">
            <div class="q-number">Question 1 of 5</div>
            <div class="q-text">What data structure does the Shunting-Yard algorithm use to temporarily hold operators during the infix-to-RPN conversion?</div>
            <div class="rpn-quiz__choices" id="kq1-choices">
                <button onclick="checkKQ(1,0)" class="rpn-quiz__choice">A) Queue</button>
                <button onclick="checkKQ(1,1)" class="rpn-quiz__choice">B) Stack</button>
                <button onclick="checkKQ(1,2)" class="rpn-quiz__choice">C) HashMap</button>
                <button onclick="checkKQ(1,3)" class="rpn-quiz__choice">D) LinkedList</button>
            </div>
            <div id="kq1-expl" class="rpn-quiz__explanation"></div>
        </div>

        <!-- Q2 -->
        <div class="rpn-quiz__question" id="kq2">
            <div class="q-number">Question 2 of 5</div>
            <div class="q-text">What is the RPN (postfix) form of <code>3 + 4 * 2</code>?</div>
            <div class="rpn-quiz__choices" id="kq2-choices">
                <button onclick="checkKQ(2,0)" class="rpn-quiz__choice">A) 3 4 + 2 *</button>
                <button onclick="checkKQ(2,1)" class="rpn-quiz__choice">B) 3 4 2 * +</button>
                <button onclick="checkKQ(2,2)" class="rpn-quiz__choice">C) + 3 * 4 2</button>
                <button onclick="checkKQ(2,3)" class="rpn-quiz__choice">D) 3 4 2 + *</button>
            </div>
            <div id="kq2-expl" class="rpn-quiz__explanation"></div>
        </div>

        <!-- Q3 -->
        <div class="rpn-quiz__question" id="kq3">
            <div class="q-number">Question 3 of 5</div>
            <div class="q-text">In the Token class, <code>BiFunction&lt;Double, Double, Double&gt;</code> is used for the <code>calculation</code> field. What type of Java feature is BiFunction?</div>
            <div class="rpn-quiz__choices" id="kq3-choices">
                <button onclick="checkKQ(3,0)" class="rpn-quiz__choice">A) Abstract class</button>
                <button onclick="checkKQ(3,1)" class="rpn-quiz__choice">B) Functional interface</button>
                <button onclick="checkKQ(3,2)" class="rpn-quiz__choice">C) Enum</button>
                <button onclick="checkKQ(3,3)" class="rpn-quiz__choice">D) Generic class</button>
            </div>
            <div id="kq3-expl" class="rpn-quiz__explanation"></div>
        </div>

        <!-- Q4 -->
        <div class="rpn-quiz__question" id="kq4">
            <div class="q-number">Question 4 of 5</div>
            <div class="q-text">During RPN evaluation of <code>5 3 + 2 *</code>, what is on the stack after processing the <code>+</code> token?</div>
            <div class="rpn-quiz__choices" id="kq4-choices">
                <button onclick="checkKQ(4,0)" class="rpn-quiz__choice">A) [5, 3, +]</button>
                <button onclick="checkKQ(4,1)" class="rpn-quiz__choice">B) [8]</button>
                <button onclick="checkKQ(4,2)" class="rpn-quiz__choice">C) [5, 8]</button>
                <button onclick="checkKQ(4,3)" class="rpn-quiz__choice">D) [3, 5]</button>
            </div>
            <div id="kq4-expl" class="rpn-quiz__explanation"></div>
        </div>

        <!-- Q5 -->
        <div class="rpn-quiz__question" id="kq5">
            <div class="q-number">Question 5 of 5</div>
            <div class="q-text">When the tokenizer encounters a <code>)</code> character, what does it do?</div>
            <div class="rpn-quiz__choices" id="kq5-choices">
                <button onclick="checkKQ(5,0)" class="rpn-quiz__choice">A) Ignores it — parentheses are not real tokens</button>
                <button onclick="checkKQ(5,1)" class="rpn-quiz__choice">B) Flushes any accumulated number, then adds ')' as a separator token</button>
                <button onclick="checkKQ(5,2)" class="rpn-quiz__choice">C) Pushes it directly onto the operator stack</button>
                <button onclick="checkKQ(5,3)" class="rpn-quiz__choice">D) Ends the tokenization process</button>
            </div>
            <div id="kq5-expl" class="rpn-quiz__explanation"></div>
        </div>

        <div class="rpn-quiz__score" id="kqScore" style="display:none;">
            <div class="score-label">Your Score</div>
            <div class="score-value" id="kqScoreVal">0 / 5</div>
            <div class="score-bar"><div class="score-fill" id="kqScoreBar" style="width:0%"></div></div>
        </div>
    </div>
</div>

<script>
(function() {
    const answers = { 1: 1, 2: 1, 3: 1, 4: 1, 5: 1 };
    const explanations = {
        1: { correct: 'The Shunting-Yard algorithm uses a <strong>Stack</strong> (LIFO) to hold operators temporarily. When a higher-precedence operator is encountered, lower-precedence operators are popped from the stack to the output.', incorrect: 'Think about LIFO (Last In, First Out) — operators need to be reordered based on precedence, and the most recent operator is compared first.' },
        2: { correct: '<code>3 4 2 * +</code> is correct! Due to precedence, <code>*</code> binds tighter: <code>4 * 2</code> becomes <code>4 2 *</code>, then <code>3 + 8</code> becomes <code>3 ... +</code>.', incorrect: 'Remember: <code>*</code> has higher precedence than <code>+</code>, so <code>4 * 2</code> is evaluated first. In RPN, the operator comes after its operands.' },
        3: { correct: '<code>BiFunction</code> is a <strong>functional interface</strong> — it has exactly one abstract method (<code>apply</code>), allowing lambda expressions like <code>(a, b) -> a + b</code>.', incorrect: '<code>BiFunction</code> is part of <code>java.util.function</code> — the key feature is that it\'s a functional interface, enabling lambda expressions.' },
        4: { correct: 'After <code>+</code>: pop 3 and 5, compute 5 + 3 = 8, push 8. Stack = <strong>[8]</strong>.', incorrect: 'When the <code>+</code> operator is encountered, it pops the top two values (3 and 5), adds them (5 + 3 = 8), and pushes the result.' },
        5: { correct: 'The tokenizer first flushes any multi-digit number being accumulated, then adds <code>)</code> as a separator token to the token list.', incorrect: 'Parentheses ARE added as tokens — they\'re needed by the Shunting-Yard algorithm to group operations correctly.' }
    };
    let score = 0;
    let answered = 0;

    window.checkKQ = function(q, sel) {
        const choices = document.querySelectorAll('#kq' + q + '-choices .rpn-quiz__choice');
        const expl = document.getElementById('kq' + q + '-expl');
        if (choices[0].classList.contains('rpn-quiz__choice--disabled')) return;
        answered++;
        const isCorrect = sel === answers[q];
        choices.forEach((c, i) => {
            c.classList.add('rpn-quiz__choice--disabled');
            if (i === answers[q]) c.classList.add('rpn-quiz__choice--correct');
        });
        if (isCorrect) {
            score++;
            expl.className = 'rpn-quiz__explanation rpn-quiz__explanation--correct';
            expl.innerHTML = '<strong style="color:#2ecc71;">Correct!</strong> ' + explanations[q].correct;
        } else {
            choices[sel].classList.add('rpn-quiz__choice--incorrect');
            expl.className = 'rpn-quiz__explanation rpn-quiz__explanation--incorrect';
            expl.innerHTML = '<strong style="color:#e74c3c;">Incorrect.</strong> ' + explanations[q].incorrect;
        }
        if (answered === 5) {
            const scoreDiv = document.getElementById('kqScore');
            scoreDiv.style.display = 'block';
            document.getElementById('kqScoreVal').textContent = score + ' / 5';
            document.getElementById('kqScoreBar').style.width = (score / 5 * 100) + '%';
        }
    };
})();
</script>


---

## Key Takeaways

### Data Structures Used

| Data Structure | Where | Purpose |
|----------------|-------|---------|
| **ArrayList** | `terms`, `rpnTerms` | Ordered storage of tokens |
| **Stack** | Shunting-yard & evaluation | LIFO for operator reordering and calculation |
| **HashMap** | `Tokens` class | O(1) lookup of operators by character |

### AP CSA Concepts Demonstrated

- **Inheritance** — `TermOrOperator extends Token`
- **Polymorphism** — numbers and operators stored in the same `ArrayList<TermOrOperator>`
- **Encapsulation** — private fields with getters
- **Functional Interfaces** — `BiFunction<Double, Double, Double>` for operator calculations
- **Algorithmic Thinking** — Dijkstra's Shunting-Yard algorithm

### Hack Ideas

1. Add support for **unary minus** (negative numbers like `-5 + 3`)
2. Add **trigonometric functions** (`sin`, `cos`, `tan`)
3. Add **error handling** for malformed expressions (mismatched parentheses, division by zero)
4. Build a **visual step-by-step debugger** that shows the stack state at each step
5. Create a **GUI calculator** that uses this engine under the hood